## In this notebook we will see the molecular classification performance of the tuned DreaMS when trained on the rest of the dataset 

### Importing and checking GPU:

In [ ]:
import json
import random
import time
import os
import platform
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

from dreams.api import PreTrainedModel
from dreams.models.dreams.dreams import DreaMS as DreaMSModel
from dreams.utils.data import SpectrumPreprocessor
from dreams.utils.dformats import DataFormatA

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "CUDA is not available. This experiment is intended to run on the GPU machine."
)

device = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)

### Configuring experiment:

In [ ]:
DATA_DIR = Path('/user/work/zy26641/casmi26/Group-Sprint-2/model_datasets/stage_3_100pct')  # set by nbparam.py for the BluePebble run

TRAIN_PATH = DATA_DIR / "train.parquet"
VALIDATION_PATH = DATA_DIR / "validation.parquet"
CLASS_MAPPING_PATH = DATA_DIR / "class_mapping.json"
EXPERIMENT_CONFIG_PATH = DATA_DIR / "experiment_config.json"

DREAMS_CHECKPOINT = Path('/user/work/zy26641/casmi26/models/ssl_model.ckpt')  # set by nbparam.py for the BluePebble run

OUTPUT_DIR = Path('/user/work/zy26641/casmi26/Group-Sprint-2/MichaelM/results_bluepebble/dreams_100pct_3epochs')  # set by nbparam.py for the BluePebble run
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Previous experiment outputs, used only for comparison if present.
EXPERIMENT_2_OUTPUT_DIR = Path('/user/work/zy26641/casmi26/Group-Sprint-2/MichaelM/for_michael_tuned_dreams/results/fine_tuned_dreams_experiment_2')  # set by nbparam.py for the BluePebble run
EXPERIMENT_2_PREDICTIONS_PATH = (
    EXPERIMENT_2_OUTPUT_DIR / "validation_predictions.csv"
)
EXPERIMENT_2_METRICS_PATH = EXPERIMENT_2_OUTPUT_DIR / "metrics.json"
EXPERIMENT_2_TRAIN_PATH = Path('/user/work/zy26641/casmi26/Group-Sprint-2/model_datasets/stage_1_5pct/train.parquet')  # set by nbparam.py for the BluePebble run

SEED = 42

N_HIGHEST_PEAKS = 60
PRECURSOR_INTENSITY = 1.1

BATCH_SIZE = 32
MAX_EPOCHS = 3  # set by nbparam.py for the BluePebble run
EARLY_STOPPING_PATIENCE = 5

ENCODER_LR = 1e-5
CLASSIFIER_LR = 3e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.3

NUM_WORKERS = 0 if os.name == "nt" else 4

EXPERIMENT_START_TIME = time.time()

print("Training data:", TRAIN_PATH)
print("Validation data:", VALIDATION_PATH)
print("Checkpoint:", DREAMS_CHECKPOINT)
print("Output directory:", OUTPUT_DIR)

### Checking input files:

In [ ]:
required_files = [
    TRAIN_PATH,
    VALIDATION_PATH,
    CLASS_MAPPING_PATH,
    EXPERIMENT_CONFIG_PATH,
    DREAMS_CHECKPOINT
]

for path in required_files:
    assert path.exists(), f"Missing file: {path.resolve()}"
    print(f"Found: {path}")

In [ ]:
train_df = pd.read_parquet(TRAIN_PATH)
validation_df_original = pd.read_parquet(VALIDATION_PATH)

training_molecules = set(
    train_df["inchikey14"].dropna().astype(str).unique()
)

validation_df = validation_df_original.copy()
validation_df["inchikey14"] = validation_df["inchikey14"].astype(str)
train_df["inchikey14"] = train_df["inchikey14"].astype(str)

validation_df = validation_df[
    validation_df["inchikey14"].isin(training_molecules)
].copy()

print(f"Training spectra: {len(train_df):,}")
print(
    "Validation spectra retained:",
    f"{len(validation_df):,} of {len(validation_df_original):,}"
)
print("Training molecules/classes:", train_df["inchikey14"].nunique())
print("Validation molecules/classes:", validation_df["inchikey14"].nunique())

assert "mzs" in train_df.columns and "intensities" in train_df.columns
assert "precursor_mz" in train_df.columns
assert train_df["inchikey14"].notna().all()
assert validation_df["inchikey14"].notna().all()

validation_only_after_filter = (
    set(validation_df["inchikey14"]) - training_molecules
)
assert len(validation_only_after_filter) == 0

### Loading class mapping:

In [ ]:
with open(CLASS_MAPPING_PATH, "r") as f:
    mapping_data = json.load(f)


def extract_class_mapping(data):
    if isinstance(data, dict):
        for key in ["class_to_idx", "class_mapping"]:
            if key in data and isinstance(data[key], dict):
                data = data[key]
                break

        if isinstance(data, dict):
            if all(isinstance(v, int) for v in data.values()):
                return {str(k): int(v) for k, v in data.items()}

            if all(str(k).isdigit() for k in data.keys()):
                return {str(v): int(k) for k, v in data.items()}

            if "classes" in data and isinstance(data["classes"], list):
                return {str(label): i for i, label in enumerate(data["classes"])}

    if isinstance(data, list):
        return {str(label): i for i, label in enumerate(data)}

    raise ValueError(
        "Could not identify the class-to-index mapping. "
        "Please inspect class_mapping.json."
    )


class_to_idx = extract_class_mapping(mapping_data)
NUM_CLASSES = len(class_to_idx)

assert sorted(class_to_idx.values()) == list(range(NUM_CLASSES))

print("Number of training classes:", NUM_CLASSES)
print("First five mapping entries:", list(class_to_idx.items())[:5])

### Checking labels and assigning class indices:

In [ ]:
unknown_training_labels = set(train_df["inchikey14"]) - set(class_to_idx)
unknown_validation_labels = set(validation_df["inchikey14"]) - set(class_to_idx)

print("Training classes:", train_df["inchikey14"].nunique())
print("Validation classes:", validation_df["inchikey14"].nunique())
print("Unknown training labels:", len(unknown_training_labels))
print("Validation-only classes after filtering:", len(unknown_validation_labels))

assert len(unknown_training_labels) == 0
assert len(unknown_validation_labels) == 0

train_df["label"] = train_df["inchikey14"].map(class_to_idx).astype(np.int64)
validation_df["label"] = (
    validation_df["inchikey14"].map(class_to_idx).astype(np.int64)
)

assert train_df["label"].min() >= 0
assert train_df["label"].max() < NUM_CLASSES

print("Label mapping checks passed.")

### Initialising DreaMS preprocessing:

In [ ]:
dformat = DataFormatA()

spec_preproc = SpectrumPreprocessor(
    dformat=dformat,
    prec_intens=PRECURSOR_INTENSITY,
    n_highest_peaks=N_HIGHEST_PEAKS,
    normalize_mzs=False,
    precision=32,
    to_relative_intensities=True
)

print("DreaMS spectrum preprocessor initialised.")

In [ ]:
def preprocess_dataframe(df, dataset_name):
    processed_spectra = []

    for i, row in enumerate(df.itertuples(index=False)):
        mzs = np.asarray(row.mzs, dtype=np.float32)
        intensities = np.asarray(row.intensities, dtype=np.float32)
        precursor_mz = float(row.precursor_mz)

        if len(mzs) != len(intensities):
            raise ValueError(f"Mismatched peak arrays in {dataset_name}, row {i}")
        if len(mzs) == 0:
            raise ValueError(f"Empty spectrum in {dataset_name}, row {i}")

        spectrum = np.vstack([mzs, intensities])

        processed = spec_preproc(
            spectrum,
            precursor_mz,
            high_form=False,
            augment=False
        )

        processed = np.asarray(processed, dtype=np.float32)

        if processed.shape != (N_HIGHEST_PEAKS + 1, 2):
            raise ValueError(
                f"Unexpected preprocessed shape at {dataset_name}, row {i}: "
                f"{processed.shape}"
            )
        if not np.isfinite(processed).all():
            raise ValueError(f"Non-finite values in {dataset_name}, row {i}")

        processed_spectra.append(processed)

        if (i + 1) % 10000 == 0:
            print(f"{dataset_name}: processed {i + 1:,} spectra")

    if not processed_spectra:
        raise ValueError(f"No spectra were processed for {dataset_name}.")

    result = np.stack(processed_spectra)
    print(f"\n{dataset_name} preprocessing complete.")
    print("Output shape:", result.shape)
    return result


preprocessing_start = time.time()
X_train = preprocess_dataframe(train_df, "Training")
training_preprocessing_seconds = time.time() - preprocessing_start

preprocessing_start = time.time()
X_validation = preprocess_dataframe(validation_df, "Validation")
validation_preprocessing_seconds = time.time() - preprocessing_start

y_train = train_df["label"].to_numpy(dtype=np.int64)
y_validation = validation_df["label"].to_numpy(dtype=np.int64)

print("\nFinal array shapes:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_validation:", X_validation.shape)
print("y_validation:", y_validation.shape)
print(
    "Preprocessing time (minutes):",
    round((training_preprocessing_seconds + validation_preprocessing_seconds) / 60, 2)
)

In [ ]:
class SpectrumClassificationDataset(Dataset):
    def __init__(self, spectra, labels):
        self.spectra = torch.tensor(spectra, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.spectra[index], self.labels[index]


train_dataset = SpectrumClassificationDataset(X_train, y_train)
validation_dataset = SpectrumClassificationDataset(X_validation, y_validation)

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    generator=loader_generator
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))

### Loading pretrained DreaMS:

In [ ]:
pretrained = PreTrainedModel.from_ckpt(
    ckpt_path=str(DREAMS_CHECKPOINT),
    ckpt_cls=DreaMSModel,
    n_highest_peaks=N_HIGHEST_PEAKS
)

dreams_encoder = pretrained.model
print("Pretrained DreaMS encoder loaded.")

### Defining the tuned model:

In [ ]:
class FineTunedDreaMSClassifier(nn.Module):
    def __init__(self, encoder, num_classes, dropout=0.3):
        super().__init__()
        self.encoder = encoder
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(1024, num_classes)

    def forward(self, x):
        embeddings = self.encoder(x)
        embeddings = embeddings[:, 0, :]
        embeddings = self.dropout(embeddings)
        return self.classifier(embeddings)


model = FineTunedDreaMSClassifier(
    encoder=dreams_encoder,
    num_classes=NUM_CLASSES,
    dropout=DROPOUT
).to(device)

for parameter in model.encoder.parameters():
    parameter.requires_grad = True

trainable_parameters = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_parameters = sum(p.numel() for p in model.parameters())

print("Model created.")
print("Number of classes:", NUM_CLASSES)
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Total parameters: {total_parameters:,}")

### Again performing a model preflight

In [ ]:
model.train()
sample_spectra, sample_labels = next(iter(train_loader))
sample_spectra = sample_spectra[:2].to(device)
sample_labels = sample_labels[:2].to(device)

model.zero_grad(set_to_none=True)
sample_logits = model(sample_spectra)

print("Input shape:", sample_spectra.shape)
print("Output shape:", sample_logits.shape)
assert sample_logits.shape == (2, NUM_CLASSES)

sample_loss = nn.CrossEntropyLoss()(sample_logits, sample_labels)
sample_loss.backward()

print("Preflight loss:", sample_loss.item())
print("Forward and backward passes successful.")

model.zero_grad(set_to_none=True)
del sample_spectra, sample_labels, sample_logits, sample_loss
torch.cuda.empty_cache()

### Setting reproducibility and optimiser (Adam) and using cross entropy loss function:

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {"params": model.encoder.parameters(), "lr": ENCODER_LR},
        {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

print("Optimiser:", optimizer)
print("Encoder learning rate:", ENCODER_LR)
print("Classifier learning rate:", CLASSIFIER_LR)
print("Weight decay:", WEIGHT_DECAY)

### Creating training and validation functions:

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (spectra, labels) in enumerate(loader):
        spectra = spectra.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(spectra)
        loss = criterion(logits, labels)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        predictions = torch.argmax(logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        if (batch_idx + 1) % 200 == 0:
            print(
                f"  Batch {batch_idx + 1}/{len(loader)} "
                f"| Loss: {loss.item():.4f}"
            )

    return running_loss / total, correct / total


def calculate_metrics(all_labels, all_predictions):
    # Preserve sklearn's original default macro-F1 definition (labels in
    # y_true or y_pred) for continuity with the previous notebook.
    return {
        "accuracy": accuracy_score(all_labels, all_predictions),
        "balanced_accuracy": balanced_accuracy_score(all_labels, all_predictions),
        "macro_f1": f1_score(
            all_labels, all_predictions, average="macro", zero_division=0
        ),
        "weighted_f1": f1_score(
            all_labels, all_predictions, average="weighted", zero_division=0
        ),
        # Supplementary metric: macro-average over classes actually present
        # in the ground-truth validation labels.
        "macro_f1_true_classes": f1_score(
            all_labels,
            all_predictions,
            labels=np.unique(all_labels),
            average="macro",
            zero_division=0
        )
    }


def validate_one_epoch(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    total = 0
    all_labels = []
    all_predictions = []
    all_confidences = []

    with torch.no_grad():
        for spectra, labels in loader:
            spectra = spectra.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(spectra)
            loss = criterion(logits, labels)

            probabilities = torch.softmax(logits, dim=1)
            confidences, predictions = torch.max(probabilities, dim=1)

            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)

            all_labels.extend(labels.cpu().numpy())
            all_predictions.extend(predictions.cpu().numpy())
            all_confidences.extend(confidences.cpu().numpy())

    labels_array = np.asarray(all_labels)
    predictions_array = np.asarray(all_predictions)
    confidence_array = np.asarray(all_confidences)

    results = {
        "loss": running_loss / total,
        **calculate_metrics(labels_array, predictions_array),
        "labels": labels_array,
        "predictions": predictions_array,
        "confidences": confidence_array
    }
    return results

### Doing the main training loop:

In [ ]:
history = []

best_validation_accuracy = -1.0
best_epoch = 0
epochs_without_improvement = 0

BEST_MODEL_PATH = Path('/user/work/zy26641/casmi26/checkpoints/dreams_100pct.pt')  # set by nbparam.py for the BluePebble run

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

training_start_time = time.time()

for epoch in range(1, MAX_EPOCHS + 1):
    print("\n" + "=" * 60)
    print(f"EPOCH {epoch}/{MAX_EPOCHS}")
    print("=" * 60)

    epoch_start = time.time()

    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )

    val_results = validate_one_epoch(
        model, validation_loader, criterion, device
    )

    epoch_time = time.time() - epoch_start

    epoch_results = {
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "validation_loss": val_results["loss"],
        "validation_accuracy": val_results["accuracy"],
        "validation_balanced_accuracy": val_results["balanced_accuracy"],
        "validation_macro_f1": val_results["macro_f1"],
        "validation_macro_f1_true_classes": val_results["macro_f1_true_classes"],
        "validation_weighted_f1": val_results["weighted_f1"],
        "epoch_time_seconds": epoch_time
    }
    history.append(epoch_results)

    print(f"\nTraining loss:       {train_loss:.4f}")
    print(f"Training accuracy:   {train_accuracy:.4f}")
    print(f"Validation loss:     {val_results['loss']:.4f}")
    print(f"Validation accuracy: {val_results['accuracy']:.4f}")
    print(f"Balanced accuracy:   {val_results['balanced_accuracy']:.4f}")
    print(f"Macro F1:            {val_results['macro_f1']:.4f}")
    print(f"Weighted F1:         {val_results['weighted_f1']:.4f}")
    print(f"Epoch time:          {epoch_time / 60:.2f} minutes")

    if val_results["accuracy"] > best_validation_accuracy:
        best_validation_accuracy = val_results["accuracy"]
        best_epoch = epoch
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "validation_accuracy": best_validation_accuracy,
            "class_to_idx": class_to_idx,
            "config": {
                "seed": SEED,
                "batch_size": BATCH_SIZE,
                "encoder_lr": ENCODER_LR,
                "classifier_lr": CLASSIFIER_LR,
                "weight_decay": WEIGHT_DECAY,
                "dropout": DROPOUT,
                "n_highest_peaks": N_HIGHEST_PEAKS,
                "precursor_intensity": PRECURSOR_INTENSITY,
                "num_classes": NUM_CLASSES
            }
        }

        torch.save(checkpoint, BEST_MODEL_PATH)
        print("\nNew best model saved.")
    else:
        epochs_without_improvement += 1
        print(
            f"\nNo validation accuracy improvement for "
            f"{epochs_without_improvement} epoch(s)."
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

total_training_time = time.time() - training_start_time

print("\n" + "=" * 60)
print("TRAINING COMPLETE")
print("=" * 60)
print("Best epoch:", best_epoch)
print(f"Best validation accuracy: {best_validation_accuracy:.4f}")
print(f"Total training time: {total_training_time / 3600:.2f} hours")
print("Best model:", BEST_MODEL_PATH)

### Restoring the best model:

In [ ]:
try:
    best_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=device,
        weights_only=True
    )
except TypeError:
    best_checkpoint = torch.load(
        BEST_MODEL_PATH,
        map_location=device
    )

model.load_state_dict(best_checkpoint["model_state_dict"])
model.to(device)

print("Best model restored.")
print("Best epoch:", best_checkpoint["epoch"])
print("Best validation accuracy:", best_checkpoint["validation_accuracy"])

### Generating metrics:

In [ ]:
final_results = validate_one_epoch(
    model, validation_loader, criterion, device
)

print("Final validation metrics:")
for metric in [
    "loss",
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "macro_f1_true_classes",
    "weighted_f1"
]:
    print(f"{metric}: {final_results[metric]:.6f}")

idx_to_class = {v: k for k, v in class_to_idx.items()}

prediction_df = pd.DataFrame({
    "true_label_index": final_results["labels"],
    "predicted_label_index": final_results["predictions"],
    "prediction_confidence": final_results["confidences"]
})

prediction_df["true_inchikey14"] = prediction_df["true_label_index"].map(idx_to_class)
prediction_df["predicted_inchikey14"] = prediction_df["predicted_label_index"].map(idx_to_class)

for column in ["source_part", "row_id"]:
    if column in validation_df.columns:
        prediction_df[column] = validation_df[column].to_numpy()

prediction_df["correct"] = (
    prediction_df["true_label_index"] == prediction_df["predicted_label_index"]
)

PREDICTIONS_PATH = OUTPUT_DIR / "validation_predictions.csv"
prediction_df.to_csv(PREDICTIONS_PATH, index=False)

print("\nPredictions saved to:", PREDICTIONS_PATH)
print("Correct predictions:", prediction_df["correct"].sum())
print("Total predictions:", len(prediction_df))

### Saving training history and experiment metrics:

In [ ]:
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if torch.cuda.is_available() else None
)

total_experiment_time = time.time() - EXPERIMENT_START_TIME

metrics = {
    "experiment": "Fine-tuned DreaMS classification — Experiment 3 scaling",
    "training_spectra": int(len(train_df)),
    "validation_spectra_before_filtering": int(len(validation_df_original)),
    "validation_spectra": int(len(validation_df)),
    "validation_classes": int(validation_df["inchikey14"].nunique()),
    "validation_only_classes_after_filtering": 0,
    "number_of_training_classes": int(NUM_CLASSES),
    "best_epoch": int(best_epoch),
    "best_validation_accuracy": float(best_validation_accuracy),
    "validation_loss": float(final_results["loss"]),
    "accuracy": float(final_results["accuracy"]),
    "balanced_accuracy": float(final_results["balanced_accuracy"]),
    "macro_f1": float(final_results["macro_f1"]),
    "macro_f1_true_classes": float(final_results["macro_f1_true_classes"]),
    "weighted_f1": float(final_results["weighted_f1"]),
    "training_preprocessing_time_seconds": float(training_preprocessing_seconds),
    "validation_preprocessing_time_seconds": float(validation_preprocessing_seconds),
    "total_training_time_hours": float(total_training_time / 3600),
    "total_experiment_time_hours": float(total_experiment_time / 3600),
    "average_epoch_time_seconds": float(history_df["epoch_time_seconds"].mean()),
    "peak_gpu_memory_allocated_gb": (
        float(peak_gpu_memory_gb) if peak_gpu_memory_gb is not None else None
    ),
    "gpu": GPU_NAME,
    "pytorch_version": torch.__version__,
    "python_version": platform.python_version(),
    "hyperparameters": {
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "encoder_lr": ENCODER_LR,
        "classifier_lr": CLASSIFIER_LR,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "gradient_clip_norm": 1.0,
        "optimiser": "AdamW",
        "loss": "CrossEntropyLoss",
        "checkpoint_selection_metric": "validation_accuracy"
    }
}

with open(OUTPUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=4)

print("Metrics and training history saved.")

### Plotting curves:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_df["epoch"], history_df["train_loss"], label="Training loss")
axes[0].plot(history_df["epoch"], history_df["validation_loss"], label="Validation loss")
axes[0].axvline(best_epoch, linestyle="--", label=f"Best epoch ({best_epoch})")
axes[0].set_title("Training and validation loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(history_df["epoch"], history_df["train_accuracy"], label="Training accuracy")
axes[1].plot(history_df["epoch"], history_df["validation_accuracy"], label="Validation accuracy")
axes[1].axvline(best_epoch, linestyle="--", label=f"Best epoch ({best_epoch})")
axes[1].set_title("Training and validation accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

### Analysing performance:

In [ ]:
class_frequency = train_df["inchikey14"].value_counts().rename("training_spectra_for_class")

frequency_predictions = prediction_df.copy()
frequency_predictions["training_spectra_for_class"] = (
    frequency_predictions["true_inchikey14"].map(class_frequency)
)

def frequency_group(n):
    if n == 1:
        return "1"
    if n == 2:
        return "2"
    if n <= 5:
        return "3–5"
    if n <= 10:
        return "6–10"
    return ">10"

frequency_predictions["training_frequency_group"] = (
    frequency_predictions["training_spectra_for_class"].map(frequency_group)
)

frequency_order = ["1", "2", "3–5", "6–10", ">10"]
frequency_summary = (
    frequency_predictions.groupby("training_frequency_group", observed=False)
    .agg(
        validation_spectra=("correct", "size"),
        correct_predictions=("correct", "sum"),
        accuracy=("correct", "mean"),
        distinct_validation_classes=("true_inchikey14", "nunique")
    )
    .reindex(frequency_order)
    .dropna(subset=["validation_spectra"])
)

display(frequency_summary)

ax = frequency_summary["accuracy"].plot(kind="bar", figsize=(9, 5))
ax.set_title("Experiment 3 accuracy by training spectra per molecule")
ax.set_xlabel("Number of training spectra for the true molecule")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.grid(axis="y")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "accuracy_by_class_frequency.png", dpi=300, bbox_inches="tight")
plt.show()

frequency_summary.to_csv(OUTPUT_DIR / "accuracy_by_class_frequency.csv")

### Analysing confidence:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(
    prediction_df.loc[prediction_df["correct"], "prediction_confidence"],
    bins=30,
    alpha=0.6,
    label="Correct predictions"
)
ax.hist(
    prediction_df.loc[~prediction_df["correct"], "prediction_confidence"],
    bins=30,
    alpha=0.6,
    label="Incorrect predictions"
)

ax.set_title("Prediction confidence: correct vs incorrect")
ax.set_xlabel("Maximum softmax probability")
ax.set_ylabel("Number of validation spectra")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prediction_confidence.png", dpi=300, bbox_inches="tight")
plt.show()

### Comparing:

In [ ]:
comparison_available = (
    EXPERIMENT_2_PREDICTIONS_PATH.exists()
    and "row_id" in prediction_df.columns
)

if comparison_available:
    exp2_predictions = pd.read_csv(EXPERIMENT_2_PREDICTIONS_PATH)

    if "row_id" not in exp2_predictions.columns:
        comparison_available = False
        print("Experiment 2 predictions do not contain row_id; shared-row comparison skipped.")
    else:
        exp2_predictions = exp2_predictions[
            ["row_id", "true_label_index", "predicted_label_index"]
        ].copy()

        exp3_predictions = prediction_df[
            ["row_id", "true_label_index", "predicted_label_index"]
        ].copy()

        if exp2_predictions["row_id"].duplicated().any() or exp3_predictions["row_id"].duplicated().any():
            raise ValueError("row_id is not unique; cannot safely join prediction files.")

        shared = exp2_predictions.merge(
            exp3_predictions,
            on="row_id",
            how="inner",
            suffixes=("_exp2", "_exp3"),
            validate="one_to_one"
        )

        if len(shared) == 0:
            comparison_available = False
            print("No shared validation row_ids found; shared-row comparison skipped.")
        else:
            # Check that both experiments refer to the same true class for each shared spectrum.
            assert (
                shared["true_label_index_exp2"] == shared["true_label_index_exp3"]
            ).all()

            shared_labels = shared["true_label_index_exp2"].to_numpy()
            exp2_pred = shared["predicted_label_index_exp2"].to_numpy()
            exp3_pred = shared["predicted_label_index_exp3"].to_numpy()

            exp2_shared_metrics = calculate_metrics(shared_labels, exp2_pred)
            exp3_shared_metrics = calculate_metrics(shared_labels, exp3_pred)

            shared_comparison = pd.DataFrame([
                {"experiment": "Experiment 2", **exp2_shared_metrics},
                {"experiment": "Experiment 3", **exp3_shared_metrics}
            ])

            print(f"Shared validation spectra: {len(shared):,}")
            display(shared_comparison)

            shared_comparison.to_csv(
                OUTPUT_DIR / "shared_validation_comparison.csv",
                index=False
            )

            metric_names = [
                "accuracy",
                "balanced_accuracy",
                "macro_f1",
                "weighted_f1"
            ]

            plot_data = shared_comparison.set_index("experiment")[metric_names]
            ax = plot_data.plot(kind="bar", figsize=(11, 6))
            ax.set_title("Experiment 2 vs Experiment 3 on shared validation spectra")
            ax.set_ylabel("Score")
            ax.set_ylim(0, 1)
            ax.set_xlabel("")
            ax.grid(axis="y")
            plt.xticks(rotation=0)
            plt.tight_layout()
            plt.savefig(
                OUTPUT_DIR / "shared_validation_metric_comparison.png",
                dpi=300,
                bbox_inches="tight"
            )
            plt.show()
else:
    print(
        "Shared validation comparison unavailable. Ensure the Experiment 2 "
        "validation_predictions.csv is in the configured results directory "
        "and contains row_id."
    )

In [ ]:
comparison_rows = [{
    "experiment": "Experiment 3",
    "training_spectra": len(train_df),
    "training_classes": NUM_CLASSES,
    "accuracy_full_eligible_validation": final_results["accuracy"],
    "balanced_accuracy_full_eligible_validation": final_results["balanced_accuracy"],
    "macro_f1_full_eligible_validation": final_results["macro_f1"],
    "weighted_f1_full_eligible_validation": final_results["weighted_f1"],
    "training_time_hours": total_training_time / 3600
}]

if EXPERIMENT_2_METRICS_PATH.exists():
    with open(EXPERIMENT_2_METRICS_PATH, "r") as f:
        exp2_metrics = json.load(f)

    comparison_rows.insert(0, {
        "experiment": "Experiment 2",
        "training_spectra": exp2_metrics.get("training_spectra"),
        "training_classes": exp2_metrics.get("number_of_training_classes"),
        "accuracy_full_eligible_validation": exp2_metrics.get("accuracy"),
        "balanced_accuracy_full_eligible_validation": exp2_metrics.get("balanced_accuracy"),
        "macro_f1_full_eligible_validation": exp2_metrics.get("macro_f1"),
        "weighted_f1_full_eligible_validation": exp2_metrics.get("weighted_f1"),
        "training_time_hours": exp2_metrics.get("total_training_time_hours")
    })

scaling_comparison = pd.DataFrame(comparison_rows)
display(scaling_comparison)
scaling_comparison.to_csv(OUTPUT_DIR / "experiment_scaling_comparison.csv", index=False)

if len(scaling_comparison) == 2:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].bar(
        scaling_comparison["experiment"],
        scaling_comparison["training_spectra"]
    )
    axes[0].set_title("Training dataset size")
    axes[0].set_ylabel("Training spectra")
    axes[0].grid(axis="y")

    axes[1].bar(
        scaling_comparison["experiment"],
        scaling_comparison["training_time_hours"]
    )
    axes[1].set_title("Training time")
    axes[1].set_ylabel("Hours")
    axes[1].grid(axis="y")

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "training_size_and_time_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Experiment 2 metrics.json not found; saved Experiment 3 metrics only.")

### Confirming experiment outputs:

In [ ]:
expected_outputs = [
    "best_finetuned_dreams.pt",
    "validation_predictions.csv",
    "training_history.csv",
    "metrics.json",
    "training_curves.png",
    "accuracy_by_class_frequency.png",
    "accuracy_by_class_frequency.csv",
    "prediction_confidence.png",
    "experiment_scaling_comparison.csv"
]

print("Experiment output files:\n")
for filename in expected_outputs:
    path = OUTPUT_DIR / filename
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"OK  {filename} ({size_mb:.2f} MB)")
    else:
        print(f"MISSING  {filename}")

print("\nExperiment 3 completed.")